# 🔬 STCL — Dual RP (Scan + Monitor separated)
**Scanning Transfer Cavity Lock** · RedPitaya STEMlab 125-14

Two RedPitayas with clearly separated responsibilities:

| Board | IP | Mode | Role |
|-------|----|------|------|
| `RP_Scan` | 192.168.0.201 | `scan` | Generates triangle ramp · locks cavity length via PID |
| `RP_Mon`  | 192.168.0.99  | `monitor` | Acquires cavity signal · runs live monitor · error monitor |

---
### 🔌 Wiring
```
RP_Scan  OUT2  ──→  piezo amp  ──→  FPI cavity input
RP_Scan  OUT2  ──→  RP_Mon IN2      (scan ramp as trigger reference)
FPI cavity output (photodiode) ──→  RP_Mon IN1  [HV mode, ±20 V]
RP_Mon   IN2   [HV mode, ±20 V]    (ramp ≤ 1 V — well within range)
```

---
### 📋 Execution order — run cells top to bottom, in sequence

| Phase | What happens | What you see |
|-------|-------------|--------------|
| 0 | Edit config | — |
| 1 | Connect boards, start event loop | Board IPs printed |
| 2 | Single acquisition sanity check | IN1 + IN2 plot |
| 3 | Push settings → start scan → open monitor | Qt window: peaks sweeping |
| 3b | *(optional)* Tune parameters live | Monitor updates |
| 4 | Start cavity lock *(blocking cell)* | Qt window: peak freezes at lockpoint |
| **4b** | **Stop the lock cleanly** | `loop_running = False` |
| **4c** | **Shift lockpoint / amplitude live while locked** | Peak glides in monitor |
| **4d** | **Lock verification tests** | PID state, step response, duration plot |
| 5 | Switch to error monitor | Qt window: flat MHz trace = locked |
| 6 | *(optional)* Laser lock | — |
| 7 | Shutdown | — |

> **Key point — Phase 4 blocks.** Run 4b / 4c / 4d in separate cells after Phase 4 has started.
> `start_lock` stops the scan internally; never call `stop_loop` manually before Phase 4.

---
## ⚙️ Phase 0: Configuration
<blockquote style="border-left:4px solid #e67e22; padding:6px 12px; background:#fdf6ec; color:#7f4f00; border-radius:4px;">
<strong>Edit only this section.</strong> All parameters flow through automatically.
</blockquote>

In [66]:
# ── Board IPs ─────────────────────────────────────────────────────────────────
RP_SCAN_IP = "192.168.0.201"   # Scan RP — generates ramp, runs cavity PID
RP_MON_IP  = "192.168.0.99"    # Monitor RP — acquires signal, displays monitor
SSH_USER   = "root"
SSH_PASS   = "root"

# ── Scan parameters (RP_Scan) ─────────────────────────────────────────────────
CAV_DEC    = 16      # decimation: 8 → 1.0 ms | 16 → 2.1 ms | 32 → 4.2 ms | 64 → 8.4 ms
CAV_AMP    = 0.5     # V — triangle ramp half-swing  (CAV_AMP + |CAV_OFFSET| ≤ 1.0 V)
CAV_OFFSET = 0.34     # V — DC offset on ramp

# ── Cavity lock parameters (RP_Scan PID) ──────────────────────────────────────
# CAV_RANGE: two time-windows [ms] bracketing the two reference peaks (FSR markers).
# Set these after looking at Phase 2 / Phase 3 — each window must contain exactly
# one clean peak. The PID drives the second peak to CAV_LOCKPOINT.
CAV_RANGE     = [[0.28, 0.40], [0.7, 0.76]] # ms — reference peak windows
CAV_LOCKPOINT = 0.73                         # ms — PID target (dashed line in monitor)                   
CAV_PID       = {"P": 0.0, "I": 2.0, "D": 0.0, "I_val": 0, "limit": [-0.99, 0.99]}

# ── Monitor display (RP_Mon) ───────────────────────────────────────────────────
# SHOW_TRIGGER = True  → dual-axis: IN1 (cavity transmission) + IN2 (scan ramp)
# SHOW_TRIGGER = False → single-axis: IN1 only
SHOW_TRIGGER = True

# ── Slave laser locks (optional — requires a third Lock RP) ───────────────────
# Uncomment and configure when adding a laser-lock RP.

# SL1_LABEL     = "Laser_A"
# SL1_RANGE     = [0.85, 1.10]   # ms
# SL1_LOCKPOINT = 0.96            # ms
# SL1_ENABLED   = True
# SL1_PID       = {"P": 0.0, "I": 0.5, "D": 0.0, "I_val": 0, "limit": [-0.99, 0.99]}

# SL2_LABEL     = "Laser_B"
# SL2_RANGE     = [0.50, 0.85]   # ms
# SL2_LOCKPOINT = 0.60            # ms
# SL2_ENABLED   = False
# SL2_PID       = {"P": 0.0, "I": 0.5, "D": 0.0, "I_val": 0, "limit": [-0.99, 0.99]}

# ── Derived (do not edit) ─────────────────────────────────────────────────────
_CAV_PERIOD_MS = 8e-9 * 16384 * CAV_DEC * 1e3
print(f"Scan period : {_CAV_PERIOD_MS:.3f} ms  (dec={CAV_DEC})")
print(f"Amp / Offset: {CAV_AMP} V / {CAV_OFFSET} V")
print(f"Cav range   : {CAV_RANGE}  lockpoint: {CAV_LOCKPOINT} ms")
print(f"Trigger view: {'ON — dual-axis (IN1 + IN2)' if SHOW_TRIGGER else 'OFF — IN1 only'}")

Scan period : 2.097 ms  (dec=16)
Amp / Offset: 0.5 V / 0.34 V
Cav range   : [[0.28, 0.4], [0.7, 0.76]]  lockpoint: 0.73 ms
Trigger view: ON — dual-axis (IN1 + IN2)


---
## 🔌 Phase 1: Import, Upload & Connect
Locates the repo, uploads RP-side scripts to both boards,
opens SSH connections, and starts the PC-side event loop.

> Run once per session. Re-running is safe — the event loop guard prevents duplicate threads.

In [67]:
# ── Imports & repo discovery ───────────────────────────────────────────────────
import sys, pathlib, threading, time
import numpy as np
import matplotlib.pyplot as plt

_here = pathlib.Path().resolve()
for _p in [_here] + list(_here.parents)[:4]:
    if (_p / "lockclient.py").exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        print("Repo root:", _p)
        break
else:
    raise FileNotFoundError("lockclient.py not found — check repo location.")

from lockclient import LockClient, RP_client, Monitor

Repo root: C:\Users\RikteemBhowmick\Projects 2025\RedPitaya Projects\RP-STCL_dual_RP


In [68]:
# ── Build RP registry ─────────────────────────────────────────────────────────
Monitor.show_trigger = SHOW_TRIGGER

RPs = {
    "RP_Scan": RP_client((RP_SCAN_IP, 5000), {}, mode="scan"),
    "RP_Mon":  RP_client((RP_MON_IP,  5000), {}, mode="monitor"),
}

print("Uploading scripts and loading settings...")
Lock = LockClient(RPs)
print("Upload complete.")
for name, rp in Lock.RPs.items():
    print(f"  {name:8s}  mode={rp.mode:8s}  addr={rp.addr[0]}")

Uploading scripts and loading settings...
Upload complete.
  RP_Scan   mode=scan      addr=192.168.0.201
  RP_Mon    mode=monitor   addr=192.168.0.99


In [69]:
# ── SSH connect both boards ────────────────────────────────────────────────────
def _wrap(fn, err):
    try: fn()
    except Exception as exc: err["exc"] = exc

def run_with_timeout(fn, timeout_s, name):
    err = {}
    t = threading.Thread(target=lambda: _wrap(fn, err), daemon=True)
    t.start(); t.join(timeout=timeout_s)
    if t.is_alive():
        raise TimeoutError(f"{name} timed out after {timeout_s} s — check board SSH / network.")
    if "exc" in err:
        raise RuntimeError(f"{name} failed: {err['exc']}")

run_with_timeout(Lock.connect_all, timeout_s=45, name="connect_all")
print("Both boards connected.")

connecting...
Both boards connected.


In [70]:
# ── Start PC-side event loop + set decimation ─────────────────────────────────
if "stcl_thread" not in globals() or not stcl_thread.is_alive():
    stcl_thread = threading.Thread(target=Lock.start, daemon=True)
    stcl_thread.start()
    time.sleep(2)
    print("Event loop started.")
else:
    print("Event loop already running — skipping.")

Lock.set_dec("RP_Scan", CAV_DEC)
print(f"Decimation set: dec={CAV_DEC}  period={_CAV_PERIOD_MS:.3f} ms")
print()
for name, rp in Lock.RPs.items():
    status = "connected" if rp.connected else "⚠ DISCONNECTED"
    print(f"  {name:8s}  {rp.addr[0]}  {status}")

Event loop started.
Decimation set: dec=16  period=2.097 ms

  RP_Scan   192.168.0.201  connected
  RP_Mon    192.168.0.99  connected


---
## 🔍 Phase 2: Signal Verification *(optional but recommended)*
Single acquisition from `RP_Mon` to confirm wiring before scanning.

**Expected:**
- `IN1` — cavity transmission peaks on a low baseline
- `IN2` — clean triangle ramp (directly from `RP_Scan OUT2`)

Use this to set `CAV_RANGE` in Phase 0 — the two peak windows must each
contain exactly one clearly visible transmission peak.

In [ ]:
acq = np.array(Lock.send("RP_Mon", "acquire"))

if acq.size == 0:
    print("⚠  No data — check RP_Mon is connected and IN1/IN2 are wired.")
else:
    t_ms, ch1, ch2 = acq[0], acq[1], acq[2]

    fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True)

    axes[0].plot(t_ms, ch1, lw=0.8, color="#4fc3f7")
    axes[0].set_ylabel("IN1 [V]")
    axes[0].set_title("RP_Mon IN1 — Cavity transmission (HV mode)")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(t_ms, ch2, lw=0.8, color="#a5d6a7")
    axes[1].set_ylabel("IN2 [V]")
    axes[1].set_title("RP_Mon IN2 — Scan ramp from RP_Scan OUT2 (trigger reference, HV mode)")
    axes[1].set_xlabel("Time [ms]")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"IN1 range: {ch1.min():.3f} … {ch1.max():.3f} V")
    print(f"IN2 range: {ch2.min():.3f} … {ch2.max():.3f} V")

---
## 📡 Phase 3: Scan + Live Monitor
Pushes settings to `RP_Scan`, starts the ramp on `OUT2`, and opens the
live monitor on `RP_Mon`.

**What you should see in the Qt window:**
- Two transmission peaks sweeping left and right as the ramp drives the cavity
- Coloured range-span markers at `CAV_RANGE` windows
- A dashed vertical line at `CAV_LOCKPOINT`

Tune `CAV_RANGE` and `CAV_LOCKPOINT` in Phase 3b until the dashed line sits
on the peak you want to lock to. Then proceed directly to Phase 4 — **do not
stop the scan or close the monitor before running Phase 4.**

In [71]:
# ── Push cavity settings to RP_Scan ───────────────────────────────────────────
Lock.update_setting("RP_Scan", "Master", "range",     CAV_RANGE)
Lock.update_setting("RP_Scan", "Master", "lockpoint", CAV_LOCKPOINT)
Lock.update_setting("RP_Scan", "Master", "enabled",   True)
Lock.update_setting("RP_Scan", "Master", "PID",       CAV_PID)
print(f"Cavity settings pushed to RP_Scan")
print(f"  range={CAV_RANGE}  lockpoint={CAV_LOCKPOINT} ms")

check if lockpoint is still fine
Cavity settings pushed to RP_Scan
  range=[[0.28, 0.4], [0.7, 0.76]]  lockpoint=0.73 ms


In [72]:
# ── Start ramp on RP_Scan ─────────────────────────────────────────────────────
Lock.start_scan("RP_Scan", amplitude=CAV_AMP, offset=CAV_OFFSET)
print(f"Scan started on RP_Scan OUT2")
print(f"  amp={CAV_AMP} V  offset={CAV_OFFSET} V  dec={CAV_DEC}  period={_CAV_PERIOD_MS:.3f} ms")
time.sleep(3)   # allow scan + monitor server (port 5066) to initialise

Scan started on RP_Scan OUT2
  amp=0.5 V  offset=0.34 V  dec=16  period=2.097 ms
connected to <socket.socket fd=2784, family=2, type=1, proto=0, laddr=('192.168.0.137', 57965), raddr=('192.168.0.201', 5065)>


In [73]:
# ── Open live monitor on RP_Mon ───────────────────────────────────────────────
# Qt window opens. You should see transmission peaks sweeping with range markers.
Lock.start_monitor("RP_Mon")
print("Monitor open — peaks should be sweeping in the Qt window.")
print("Tune CAV_RANGE / CAV_LOCKPOINT in Phase 3b, then run Phase 4.")

Starting background process
monitoring process started
Monitor open — peaks should be sweeping in the Qt window.
Tune CAV_RANGE / CAV_LOCKPOINT in Phase 3b, then run Phase 4.


connected to <socket.socket fd=3180, family=2, type=1, proto=0, laddr=('192.168.0.137', 57970), raddr=('192.168.0.99', 5065)>


In [ ]:
# Lock.stop_monitor("RP_Mon")
# time.sleep(0.5)

### 3b 🔧 Tune scan parameters live
Edit values and **re-run this cell** at any time while the scan is running.
The monitor updates immediately — no restart needed.

In [74]:
# ── Edit here, then re-run ────────────────────────────────────────────────────
CAV_DEC       = 16                            # scan period
CAV_AMP       = 0.56                           # V — ramp amplitude
CAV_OFFSET    = 0.32                           # V — ramp DC offset
CAV_RANGE     = [[0.01, 0.2], [0.29, 0.49]] # ms — reference peak windows
CAV_LOCKPOINT = 0.357                         # ms — PID target (dashed line in monitor)  

_CAV_PERIOD_MS = 8e-9 * 16384 * CAV_DEC * 1e3

Lock.set_dec("RP_Scan", CAV_DEC)
if Lock.RPs["RP_Scan"].loop_running:
    Lock.set_scan_output("RP_Scan", amplitude=CAV_AMP, offset=CAV_OFFSET)
Lock.update_setting("RP_Scan", "Master", "range",     CAV_RANGE)
Lock.update_setting("RP_Scan", "Master", "lockpoint", CAV_LOCKPOINT)

print(f"Updated — dec={CAV_DEC}  period={_CAV_PERIOD_MS:.3f} ms")
print(f"          amp={CAV_AMP} V  offset={CAV_OFFSET} V")
print(f"          range={CAV_RANGE}  lockpoint={CAV_LOCKPOINT} ms")

check if lockpoint is still fine
Updated — dec=16  period=2.097 ms
          amp=0.56 V  offset=0.32 V
          range=[[0.01, 0.2], [0.29, 0.49]]  lockpoint=0.357 ms


---
## 🔒 Phase 4: Cavity Lock

This cell stops the free-running scan internally, engages the PID on `RP_Scan`,
and **blocks** while the lock runs.

- The ramp on `OUT2` stays active — the PID adjusts its DC offset to pin the cavity resonance at `CAV_LOCKPOINT`
- The monitor window stays open — watch the peak stop sweeping and freeze at the dashed lockpoint line

> **Do not wait for this cell to return before running sub-phases below.**
> Run Phase 4b / 4c / 4d in separate cells while this one is still blocking.

### Sub-phases (run in separate cells while Phase 4 is blocking)

| Cell | Purpose |
|------|---------|
| **4b** | Stop the lock cleanly |
| **4c** | Shift offset / lockpoint live while locked |
| **4d** | Lock verification tests |

In [78]:
# ── Refresh settings and start cavity lock ────────────────────────────────────
# start_lock() stops the scan loop internally before engaging the PID.
# Do NOT call stop_loop() manually before this cell — it would kill the ramp.
Lock.update_setting("RP_Scan", "Master", "range",     CAV_RANGE)
Lock.update_setting("RP_Scan", "Master", "lockpoint", CAV_LOCKPOINT)
Lock.update_setting("RP_Scan", "Master", "enabled",   True)
Lock.update_setting("RP_Scan", "Master", "PID",       CAV_PID)
print(f"Starting cavity lock — target: {CAV_LOCKPOINT} ms")
print("Monitor window stays open: watch the peak stop sweeping and freeze at the lockpoint.")
print("Run Phase 5 in the next cell to switch to the error monitor.")
print()

Lock.start_lock("RP_Scan")   # blocks while lock is running

# The lines below execute only after the lock is stopped (Phase 7 / Lock.close())
print("Cavity lock stopped.")

check if lockpoint is still fine
Starting cavity lock — target: 0.357 ms
Monitor window stays open: watch the peak stop sweeping and freeze at the lockpoint.
Run Phase 5 in the next cell to switch to the error monitor.

Cavity lock stopped.


connected to <socket.socket fd=2908, family=2, type=1, proto=0, laddr=('192.168.0.137', 58609), raddr=('192.168.0.201', 5065)>


### 4b ⏹ Stop the lock (scan turns off)

Run this cell while Phase 4 is still blocking to stop the PID cleanly.

`stop_loop` sends "stop" directly to the lock's port-5065 socket using raw I/O,
bypassing the shared `Sender.sel` selector. This avoids the Windows SelectSelector
deadlock that caused the previous `KeyboardInterrupt`.

In [76]:
# ── 4b: Stop the cavity lock ──────────────────────────────────────────────────
# Run while Phase 4 is blocking.  Uses raw socket I/O on port 5065 — does NOT
# touch Sender.sel, so no deadlock even with the event_loop running in parallel.

if not Lock.RPs["RP_Scan"].loop_running:
    print("Lock is not running.")
else:
    result = Lock.stop_loop("RP_Scan")
    time.sleep(0.5)
    print(f"Lock stopped.  Board reply: {result}")
    print(f"loop_running = {Lock.RPs['RP_Scan'].loop_running}")

Lock stopped.  Board reply: Stopped!
loop_running = False


### 4b-1 🔁 Restart scan + monitor after stopping lock
Run this cell after **4b** to resume the free-running scan and reopen the
cavity monitor.  Use it to compare the cavity signal **before vs after** locking
(e.g. check that the locked peak position is stable, or that no new noise was
introduced by the PID).

> The monitor window will show the scan sweeping again with the same range
> markers and lockpoint line — watch whether the peak position has drifted.

In [77]:
# ── 4b-1: Restart scan + monitor after lock stop ─────────────────────────────
# Run after Phase 4b to compare cavity signal before vs after locking.
# Restarts the triangle ramp on RP_Scan and reopens the cavity monitor.

if Lock.RPs["RP_Scan"].loop_running:
    print("Lock is still running — stop it first (run Phase 4b).")
else:
    # Close any open monitor before restarting scan
    if Lock.monitors["RP_Mon"]["running"].value or Lock.monitors["RP_Mon"]["running_err"].value:
        Lock.stop_monitor("RP_Mon")
        time.sleep(0.5)

    # Restart scan ramp on RP_Scan
    Lock.update_setting("RP_Scan", "Master", "range",     CAV_RANGE)
    Lock.update_setting("RP_Scan", "Master", "lockpoint", CAV_LOCKPOINT)
    Lock.update_setting("RP_Scan", "Master", "enabled",   True)
    Lock.update_setting("RP_Scan", "Master", "PID",       CAV_PID)
    Lock.start_scan("RP_Scan", amplitude=CAV_AMP, offset=CAV_OFFSET)
    print(f"Scan restarted on RP_Scan  (amp={CAV_AMP} V  offset={CAV_OFFSET} V)")
    time.sleep(3)   # allow scan + port 5066 to initialise

    # Reopen cavity monitor on RP_Mon
    Lock.start_monitor("RP_Mon")
    print("Monitor reopened — compare peak position to before locking.")

check if lockpoint is still fine
Scan restarted on RP_Scan  (amp=0.56 V  offset=0.32 V)
connected to <socket.socket fd=3120, family=2, type=1, proto=0, laddr=('192.168.0.137', 58495), raddr=('192.168.0.201', 5065)>
Starting background process
monitoring process started
Monitor reopened — compare peak position to before locking.


connected to <socket.socket fd=3056, family=2, type=1, proto=0, laddr=('192.168.0.137', 58501), raddr=('192.168.0.99', 5065)>


### 4c 🎚️ Live offset / lockpoint control while locked

**How offset and lockpoint interact during lock:**

The PID drives `gen_ramp.offset` (the DC level of OUT2) to keep the cavity peak
at `CAV_LOCKPOINT`. Changing the lockpoint is equivalent to shifting the scan
centre — the PID integrator is adjusted by the same delta so the transition is
bumpless (no sudden kick to the piezo).

What to watch in the monitor window:
- `shift_lockpoint` → dashed lockpoint line moves, peak follows smoothly
- `set_lock_amplitude` → scan window widens/narrows, peak stays fixed

> Run individual cells independently. Each command takes effect within one scan cycle (~2 ms at dec=16).

In [41]:
# ── 4c-1: Shift the lockpoint while locked ───────────────────────────────────
# Edit NEW_LOCKPOINT_MS and re-run.  The peak glides to the new position.
# Monitor marker updates immediately on the PC side.
# PID integrator is bumped by the same delta → bumpless transition.

NEW_LOCKPOINT_MS = CAV_LOCKPOINT   # ← edit this (must stay inside CAV_RANGE[1])

if not Lock.RPs["RP_Scan"].loop_running:
    print("Lock is not running — start Phase 4 first.")
else:
    result = Lock.shift_lockpoint("RP_Scan", NEW_LOCKPOINT_MS)
    CAV_LOCKPOINT = NEW_LOCKPOINT_MS   # keep notebook variable in sync
    print(f"Lockpoint shifted to {NEW_LOCKPOINT_MS:.4f} ms")
    print(f"Board reply: {result}")

Lockpoint shifted to 0.3570 ms
Board reply: lockpoint shifted to 0.3570 ms


In [64]:
# ── 4c-2: Sweep lockpoint in steps to trace cavity response ──────────────────
# Moves the lockpoint in a sequence of steps and pauses between each.
# Watch the monitor: the peak should track each step.
# Useful to confirm the lock is actively following and not stuck at one position.

import numpy as np, time

_lp_start = CAV_LOCKPOINT - 0.03   # ms — sweep start (relative to current lockpoint)
_lp_end   = CAV_LOCKPOINT + 0.03   # ms — sweep end
_n_steps  = 7                       # number of steps
_hold_s   = 1.5                     # seconds to hold at each step

if not Lock.RPs["RP_Scan"].loop_running:
    print("Lock is not running — start Phase 4 first.")
else:
    steps = np.linspace(_lp_start, _lp_end, _n_steps)
    print(f"Sweeping lockpoint: {_lp_start:.4f} → {_lp_end:.4f} ms  ({_n_steps} steps, {_hold_s} s each)")
    for lp in steps:
        Lock.shift_lockpoint("RP_Scan", float(lp))
        print(f"  lockpoint = {lp:.4f} ms")
        time.sleep(_hold_s)
    # Return to original lockpoint
    Lock.shift_lockpoint("RP_Scan", CAV_LOCKPOINT)
    print(f"Returned to {CAV_LOCKPOINT:.4f} ms")

Sweeping lockpoint: 0.3270 → 0.3870 ms  (7 steps, 1.5 s each)
  lockpoint = 0.3270 ms
  lockpoint = 0.3370 ms
  lockpoint = 0.3470 ms
  lockpoint = 0.3570 ms
  lockpoint = 0.3670 ms
  lockpoint = 0.3770 ms
  lockpoint = 0.3870 ms
Returned to 0.3570 ms


In [ ]:
# ── 4c-3: Change ramp amplitude while locked ─────────────────────────────────
# Narrows or widens the scan window around the locked peak.
# The PID DC offset is unaffected — only the triangle swing changes.
# Narrowing confirms the peak is truly pinned (not just within a wide window).

NEW_AMP = 0.2   # V — half-swing of OUT2 triangle  (must satisfy NEW_AMP + |CAV_OFFSET| ≤ 1.0 V)

if not Lock.RPs["RP_Scan"].loop_running:
    print("Lock is not running — start Phase 4 first.")
else:
    result = Lock.set_lock_amplitude("RP_Scan", NEW_AMP)
    print(f"Amplitude set to {NEW_AMP} V")
    print(f"Board reply: {result}")

### 4d 🧪 Lock verification tests

Run these cells while Phase 4 is still blocking (lock running).

| Test | What it checks | Pass criterion |
|------|---------------|----------------|
| **PID state** | Integrator not at rail, PID on | `MV` well inside `±0.99`, `on=True` |
| **Step response** | Peak tracks a sudden ±Δ lockpoint kick | Peak returns to lockpoint within ~1 s |
| **Perturbation rejection** | Peak stays put when ramp amp is briefly halved | Peak offset < 0.5 × FWHM |
| **Lock duration** | No unlocks over 60 s | Zero "skipped point!" messages |

In [44]:
# ── Test 1: PID state readout ─────────────────────────────────────────────────
# Reads the current PID integrator value, output (MV), limits, and on/off state.
# A healthy lock has MV well inside limits and I_val growing slowly.
# If MV is at ±0.99 the PID is railed — reduce I gain or check ramp amplitude.

if not Lock.RPs["RP_Scan"].loop_running:
    print("Lock not running.")
else:
    state = Lock._raw_loop_send("RP_Scan", "get_pid_state", "")
    if state is None:
        print("No response — board may be busy during setup_lock(). Try again in 2 s.")
    else:
        print("PID state:")
        for ch, s in state.items():
            mv    = s["MV"]
            i_val = s["I_val"]
            lim   = s["limit"]
            on    = s["on"]
            headroom = min(abs(mv - lim[0]), abs(mv - lim[1]))
            status = "OK" if headroom > 0.05 else "⚠ NEAR RAIL"
            print(f"  {ch:8s}  MV={mv:+.4f}  I_val={i_val:+.4f}  "
                  f"limits={lim}  on={on}  headroom={headroom:.3f}  {status}")

PID state:
  Master    MV=-0.0156  I_val=-0.0156  limits=[-0.99, 0.99]  on=True  headroom=0.974  OK


In [45]:
# ── Test 2: Step-response test ────────────────────────────────────────────────
# Kicks the lockpoint by ±STEP_MS and reads the PID MV before and after.
# A working lock shows MV changing to absorb the kick, then settling back.
# Watch the monitor: peak should jump, then return to lockpoint within ~1 s.

STEP_MS = 0.02   # ms — kick size (keep < half the range width)

if not Lock.RPs["RP_Scan"].loop_running:
    print("Lock not running.")
else:
    def _read_mv():
        s = Lock._raw_loop_send("RP_Scan", "get_pid_state", "")
        return s["Master"]["MV"] if s else None

    mv0 = _read_mv()
    print(f"Before kick:  MV = {mv0:+.4f}")

    # Apply positive kick
    Lock.shift_lockpoint("RP_Scan", CAV_LOCKPOINT + STEP_MS)
    time.sleep(0.1)
    mv_kick = _read_mv()
    print(f"After +{STEP_MS:.3f} ms kick:  MV = {mv_kick:+.4f}  (delta = {mv_kick - mv0:+.4f})")

    time.sleep(1.5)
    mv_settled = _read_mv()
    print(f"After 1.5 s:  MV = {mv_settled:+.4f}")

    # Restore
    Lock.shift_lockpoint("RP_Scan", CAV_LOCKPOINT)
    time.sleep(1.0)
    mv_final = _read_mv()
    print(f"Restored:     MV = {mv_final:+.4f}")

    # Verdict
    responded = abs(mv_kick - mv0) > 0.002
    print()
    print("Step response:", "PASS — PID responded to kick" if responded
          else "⚠ FAIL — MV did not change; check PID gains and ramp output")

Before kick:  MV = -0.0150
After +0.020 ms kick:  MV = +0.0020  (delta = +0.0170)
After 1.5 s:  MV = -0.0368
Restored:     MV = -0.0155

Step response: PASS — PID responded to kick


In [79]:
# ── Test 3: Perturbation rejection ───────────────────────────────────────────
# Halves the scan amplitude for 3 s, then restores it.
# A locked peak should barely move because the PID holds the DC offset.
# If the peak drifts significantly when amplitude changes, the lock is marginal
# (the peak finder may be losing track of the narrowed window).

AMP_REDUCED = CAV_AMP * 0.5   # V — half the normal scan amplitude
AMP_HOLD_S  = 3.0              # s — how long to hold the reduced amplitude

if not Lock.RPs["RP_Scan"].loop_running:
    print("Lock not running.")
else:
    mv_before = Lock._raw_loop_send("RP_Scan", "get_pid_state", "")
    mv_before = mv_before["Master"]["MV"] if mv_before else None

    print(f"Reducing amplitude: {CAV_AMP:.3f} V → {AMP_REDUCED:.3f} V")
    Lock.set_lock_amplitude("RP_Scan", AMP_REDUCED)

    time.sleep(AMP_HOLD_S)
    mv_during = Lock._raw_loop_send("RP_Scan", "get_pid_state", "")
    mv_during = mv_during["Master"]["MV"] if mv_during else None

    print(f"Restoring amplitude to {CAV_AMP:.3f} V")
    Lock.set_lock_amplitude("RP_Scan", CAV_AMP)
    time.sleep(1.0)

    if mv_before is not None and mv_during is not None:
        delta_mv = abs(mv_during - mv_before)
        print(f"\nMV before: {mv_before:+.4f}  during: {mv_during:+.4f}  delta: {delta_mv:.4f}")
        print("Perturbation rejection:", "PASS — MV stable" if delta_mv < 0.05
              else "⚠ CHECK — MV shifted; peak finder may lose track at reduced amplitude")

Reducing amplitude: 0.560 V → 0.280 V
Restoring amplitude to 0.560 V

MV before: -0.0090  during: -0.1390  delta: 0.1301
Perturbation rejection: ⚠ CHECK — MV shifted; peak finder may lose track at reduced amplitude


In [80]:
# ── Test 4: Lock duration + MV drift monitor ─────────────────────────────────
# Polls PID state every POLL_S seconds for DURATION_S seconds.
# Prints a warning if the lock drops (loop_running goes False) or if MV
# approaches the rail (headroom < 0.1).
# After the run, plots MV vs time so you can see slow cavity drift.

DURATION_S = 60.0   # s — how long to monitor
POLL_S     = 1.0    # s — polling interval

import time as _time_mod

if not Lock.RPs["RP_Scan"].loop_running:
    print("Lock not running.")
else:
    print(f"Monitoring lock for {DURATION_S:.0f} s  (polling every {POLL_S} s) ...")
    _times, _mvs = [], []
    _t0 = _time_mod.perf_counter()
    _unlocks = 0

    while _time_mod.perf_counter() - _t0 < DURATION_S:
        _t = _time_mod.perf_counter() - _t0
        if not Lock.RPs["RP_Scan"].loop_running:
            print(f"  t={_t:.1f} s  ⚠ LOCK DROPPED")
            _unlocks += 1
            break
        _s = Lock._raw_loop_send("RP_Scan", "get_pid_state", "")
        if _s and "Master" in _s:
            _mv   = _s["Master"]["MV"]
            _lim  = _s["Master"]["limit"]
            _room = min(abs(_mv - _lim[0]), abs(_mv - _lim[1]))
            _times.append(_t)
            _mvs.append(_mv)
            _warn = "  ⚠ NEAR RAIL" if _room < 0.1 else ""
            print(f"  t={_t:5.1f} s  MV={_mv:+.4f}  headroom={_room:.3f}{_warn}")
        _time_mod.sleep(POLL_S)

    print(f"\nDone.  {len(_times)} samples, {_unlocks} unlock event(s).")

    if len(_times) > 1:
        fig, ax = plt.subplots(figsize=(10, 3))
        ax.plot(_times, _mvs, lw=1.2, color="#4fc3f7")
        ax.axhline( 0.99, color="#ef9a9a", lw=0.8, ls="--", label="rail")
        ax.axhline(-0.99, color="#ef9a9a", lw=0.8, ls="--")
        ax.set_xlabel("Time [s]")
        ax.set_ylabel("PID MV  [V]")
        ax.set_title("Cavity lock — PID output (MV) vs time")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        drift = max(_mvs) - min(_mvs)
        print(f"MV range: {min(_mvs):+.4f} … {max(_mvs):+.4f}  (drift = {drift:.4f} V)")
        print("Duration test:", "PASS" if _unlocks == 0 else f"⚠ FAIL — {_unlocks} unlock(s)")

Monitoring lock for 60 s  (polling every 1.0 s) ...
  t=  0.0 s  MV=-0.0087  headroom=0.981
  t=  1.0 s  MV=-0.0094  headroom=0.981
  t=  2.0 s  MV=-0.0096  headroom=0.980
  t=  3.0 s  MV=-0.0096  headroom=0.980
  t=  4.1 s  MV=-0.0095  headroom=0.980
  t=  5.1 s  MV=-0.0092  headroom=0.981
  t=  6.1 s  MV=-0.0089  headroom=0.981
  t=  7.1 s  MV=-0.0087  headroom=0.981
  t=  8.1 s  MV=-0.0097  headroom=0.980
  t=  9.1 s  MV=-0.0094  headroom=0.981
  t= 10.2 s  MV=-0.0098  headroom=0.980
  t= 11.2 s  MV=-0.0100  headroom=0.980
  t= 12.2 s  MV=-0.0102  headroom=0.980
  t= 13.2 s  MV=-0.0098  headroom=0.980
  t= 14.2 s  MV=-0.0098  headroom=0.980
  t= 15.2 s  MV=-0.0095  headroom=0.980
  t= 16.3 s  MV=-0.0101  headroom=0.980
  t= 17.3 s  MV=-0.0087  headroom=0.981
  t= 18.3 s  MV=-0.0094  headroom=0.981
  t= 19.3 s  MV=-0.0093  headroom=0.981
  t= 20.3 s  MV=-0.0097  headroom=0.980
  t= 21.3 s  MV=-0.0087  headroom=0.981
  t= 22.3 s  MV=-0.0100  headroom=0.980
  t= 23.4 s  MV=-0.0095  hea

---
## 📊 Phase 5: Error Monitor
Run this cell **while Phase 4 is still blocking** (lock loop running).

Switches `RP_Mon` from the peak-position view to a frequency-error time trace
(MHz vs time). A flat line near 0 MHz means the cavity is locked.

**How to read it:**
- Flat, low noise around 0 MHz → cavity locked, PID holding
- Slow drift → PID integral gain too low, or lock losing
- Sudden jumps → peak escaped the range (cavity unlocked)

In [48]:
# ── Switch to error monitor ───────────────────────────────────────────────────
# Close the peak monitor first, then open the error monitor.
# Both run on RP_Mon — only one can be active at a time.
Lock.stop_monitor("RP_Mon")
time.sleep(0.5)

Lock.start_error_monitor("RP_Mon", tmin=20e-3)
print("Error monitor open on RP_Mon.")
print("Flat trace near 0 MHz = cavity locked.")
print("To stop error monitor: Lock.stop_monitor('RP_Mon')")

Starting background process
monitoring process started
Error monitor open on RP_Mon.
Flat trace near 0 MHz = cavity locked.
To stop error monitor: Lock.stop_monitor('RP_Mon')


In [49]:
# To stop the monitor, run this
Lock.stop_monitor('RP_Mon')

In [ ]:
# ── Save error trace to JSON (optional) ───────────────────────────────────────
# Uncomment and run to save the currently recorded error trace to disk.

# import time as _t
# filename = "lock_errors_{}".format(int(_t.time()))
# Lock.monitors["RP_Mon"]["queue_err"].put(("save", filename))
# print("Saved →", filename + ".json")

---
## 🔒 Phase 6: Laser Lock *(optional — requires a third Lock RP)*
Uncomment the slave config in Phase 0, add the Lock1 RP to the registry in
Phase 1, then run this cell while the cavity lock (Phase 4) is still running.

In [ ]:
# ── Push slave settings and start laser lock ───────────────────────────────────
if "Lock1" not in Lock.RPs:
    print("Lock1 not present — add RP_client for Lock1 in Phase 1, then re-run.")
else:
    Lock.update_setting("Lock1", "Slave1", "label",     SL1_LABEL)
    Lock.update_setting("Lock1", "Slave1", "range",     SL1_RANGE)
    Lock.update_setting("Lock1", "Slave1", "lockpoint", SL1_LOCKPOINT)
    Lock.update_setting("Lock1", "Slave1", "enabled",   SL1_ENABLED)
    Lock.update_setting("Lock1", "Slave1", "PID",       SL1_PID)

    Lock.update_setting("Lock1", "Slave2", "label",     SL2_LABEL)
    Lock.update_setting("Lock1", "Slave2", "range",     SL2_RANGE)
    Lock.update_setting("Lock1", "Slave2", "lockpoint", SL2_LOCKPOINT)
    Lock.update_setting("Lock1", "Slave2", "enabled",   SL2_ENABLED)
    Lock.update_setting("Lock1", "Slave2", "PID",       SL2_PID)

    Lock.start_lock("Lock1")
    print(f"Laser lock started — Slave1: {SL1_LABEL}  Slave2: {SL2_LABEL}")
    print("To stop: Lock.stop_loop('Lock1')")

---
## 🛑 Phase 7: Safe Shutdown
`Lock.close()` stops everything in the correct order:
monitors → lock loops → scan → disconnect.

Run this when you are done. It will also unblock Phase 4 if still running.

In [81]:
# ── Full automatic shutdown ───────────────────────────────────────────────────
Lock.close()
print("All monitors stopped, all loops halted, all boards disconnected.")

Main: Error: Exception for ('192.168.0.201', 5000):
Traceback (most recent call last):
  File "C:\Users\RikteemBhowmick\Projects 2025\RedPitaya Projects\RP-STCL_dual_RP\communication.py", line 126, in event_loop
    message.process_events(
  File "C:\Users\RikteemBhowmick\Projects 2025\RedPitaya Projects\RP-STCL_dual_RP\libclient.py", line 100, in process_events
    self.read()
  File "C:\Users\RikteemBhowmick\Projects 2025\RedPitaya Projects\RP-STCL_dual_RP\libclient.py", line 106, in read
    self._read()
  File "C:\Users\RikteemBhowmick\Projects 2025\RedPitaya Projects\RP-STCL_dual_RP\libclient.py", line 39, in _read
    data = self.sock.recv(self.buffersize)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host

[DEBUG] No response received from ('192.168.0.201', 5000) for action. Server may have closed connection early.
All monitors stopped, all loops halted, all boards disconnected.


In [ ]:
# ── Manual step-by-step shutdown (if needed for debugging) ────────────────────
# Lock.stop_monitor("RP_Mon")
# time.sleep(0.5)
#
# if "Lock1" in Lock.RPs and Lock.RPs["Lock1"].loop_running:
#     Lock.stop_loop("Lock1")
#     time.sleep(0.5)
#
# if Lock.RPs["RP_Scan"].loop_running:
#     Lock.stop_loop("RP_Scan")
#     time.sleep(0.5)
#
# Lock.close()
# print("Manual shutdown complete.")